# 19 — Identification at scale + verification (public mouse-dynamics corpora)

**Phase 6** (`docs/ROADMAP.md`). Answers the killer question — *does the windowed-feature identifier survive beyond 3 friends?* — on **Balabit** (10 users) and **SapiMouse** (120), and reframes identification as the actual industry problem: **verification / open-set** (account-sharing / smurf detection).

> **Status: SKELETON.** Section headers + stub cells. The per-corpus adapters (`pipeline/external/balabit.py`, `sapimouse.py`) are stubs awaiting the dataset CSVs. Fill top-to-bottom once the data is downloaded.

**Mouse-only note:** these corpora have no keyboard channel, so we train/evaluate on `MOUSE_ID_FEATURE_COLS` (keyboard features *excluded*, not zero-filled). GTA accuracy may not directly compare — that's expected and honest.

## 0. Setup

In [ ]:
from pipeline.features.run import MOUSE_ID_FEATURE_COLS

# from pipeline.external.balabit import BalabitAdapter
# from pipeline.external.sapimouse import SapiMouseAdapter

print(f"{len(MOUSE_ID_FEATURE_COLS)} mouse-only ID features:", MOUSE_ID_FEATURE_COLS)

## 1. Adapt the corpora → recorder schema → features

Each adapter yields recorder-schema session dicts; run them through the existing ingestion + feature stages so nothing downstream changes. TODO: point at the downloaded dataset dirs, export to `data/external/<corpus>/`, run `pipeline.ingestion.run` + `pipeline.features.run` on them (or call the in-memory helpers).

In [ ]:
# TODO(Phase 6): once BalabitAdapter/SapiMouseAdapter parse the CSVs —
#   adapter = BalabitAdapter(src=Path('data/external/balabit'))
#   adapter.export(Path('data/external/balabit_json'))
#   -> ingest -> features  (mouse-only; keyboard features will be NaN/absent)
# Sanity: every exported session passes ingestion.run.validate_session.

## 2. Closed-set identification — the users-curve

Accuracy (with bootstrap CIs) vs number of enrolled users: **3 → 10 → 50 → 120**. Session-held-out splits (reuse `pipeline/features/split.py`'s per-user holdout). The headline plot is accuracy-vs-N with the chance line; the question it answers is whether the method degrades gracefully or collapses past a handful of users.

In [ ]:
# TODO: train LightGBM on MOUSE_ID_FEATURE_COLS at each N, bootstrap-CI the
# test accuracy, plot accuracy vs N. Write reports/external_identification.json
# (consumed by scripts/generate_results.py for the README row).

## 3. Verification — ROC / EER / DET (the product reframe)

Convert the closed-set model to **pairwise verification**: per-user one-vs-rest scores → ROC, **Equal Error Rate**, DET curve. This is the smurf / account-sharing framing. `pipeline/verification.py` (to build) holds the scoring + EER so it's unit-tested and reusable.

In [ ]:
# TODO: from pipeline.verification import eer, det_curve
# genuine vs impostor score distributions -> EER + DET. Balabit's is_illegal
# test labels give real impostor sessions directly.

## 4. Open-set rejection

Enrol K users, hold the rest out as **impostors the model has never seen**. Report false-accept at a fixed rejection threshold ("none of the enrolled players"). This is what separates a toy closed-set classifier from something deployable.

In [ ]:
# TODO: max-softmax / score-threshold rejection; FAR @ fixed FRR; sweep the
# enrolled-vs-unknown split.

## 5. Findings

Write the honest result here (→ `docs/VERIFICATION.md`, `docs/FINDINGS.md`, `docs/REPORT.md` §scale-up + §verification):
- closed-set accuracy at 10 / 120 users vs the 3-player GTA number;
- EER on Balabit (literature-comparable);
- open-set FAR@FRR;
- mouse-only caveat (no keyboard channel) stated plainly.

A drop vs GTA is a *finding*, not a failure — it isolates how much of the GTA number was keyboard-driven.